In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np

In [ ]:
def preprocess_text(text):
    text = re.sub(r'[^\w\s]', '', str(text).lower().strip())
    return text

In [ ]:
df = pd.read_csv('Twitter_data.csv')
df['clean_text'] = df['clean_text'].apply(preprocess_text)
valid_categories = [-1, 0, 1]
df = df[df['category'].isin(valid_categories)]
category_mapping = {-1: 0, 0: 1, 1: 2}
df['category'] = df['category'].map(category_mapping)

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
max_len = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')
labels = to_categorical(df['category'], num_classes=3)

In [ ]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len))
model.add(LSTM(units=128))
model.add(Dense(units=3, activation='softmax'))
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(padded_sequences, labels, epochs=10, batch_size=32, validation_split=0.2)

In [ ]:
model.save_weights('sentiment_model1.weights.h5')
print("Model weights saved successfully.")

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import re
import numpy as np

import faiss
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras import Input

df = pd.read_csv('Twitter_data.csv')
def preprocess_text(text):
    return re.sub(r'[^\w\s]', '', str(text).lower().strip())
df['clean_text'] = df['clean_text'].apply(preprocess_text)
valid_categories = [-1, 0, 1]
df = df[df['category'].isin(valid_categories)]
category_mapping = {-1: 0, 0: 1, 1: 2}
df['category'] = df['category'].map(category_mapping)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['clean_text'])
sequences = tokenizer.texts_to_sequences(df['clean_text'])
max_len = max(len(seq) for seq in sequences)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100
model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len, name="embedding_layer"))
model.add(LSTM(units=128))
model.add(Dense(units=3, activation='softmax'))
model.build((None, max_len))
model.load_weights('sentiment_model1.weights.h5')
print("Model weights loaded successfully.")
input_tensor = Input(shape=(max_len,))
embedding_output = model.layers[0](input_tensor)
embedding_model = Model(inputs=input_tensor, outputs=embedding_output)
embeddings = embedding_model.predict(padded_sequences)
avg_embeddings = np.mean(embeddings, axis=1).astype('float32')
index = faiss.IndexFlatL2(embedding_dim)
index.add(avg_embeddings)
print("FAISS index built with", index.ntotal, "vectors.")
sentiment_labels = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
def predict_sentiment(input_text):
    processed_text = preprocess_text(input_text)
    seq = tokenizer.texts_to_sequences([processed_text])
    padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
    prediction = model.predict(padded_seq)
    predicted_class = np.argmax(prediction, axis=1)[0]
    predicted_sentiment = sentiment_labels[predicted_class]
    input_embedding = embedding_model.predict(padded_seq)
    input_avg_embedding = np.mean(input_embedding, axis=1).astype('float32')
    k = 3
    distances, indices = index.search(input_avg_embedding, k)
    similar_texts = df.iloc[indices[0]]
    similar_info = []
    for idx, row in similar_texts.iterrows():
        sim_text = row['clean_text']
        sim_sentiment = sentiment_labels[row['category']]
        similar_info.append(f"Text: '{sim_text}', Sentiment: {sim_sentiment}")
    print(f"DEBUG: Predicted class = {predicted_class}, Probabilities = {prediction}")
    print("Similar texts retrieved from FAISS vector database:")
    for info in similar_info:
        print(info)
    return predicted_sentiment
while True:
    user_input = input("Enter text (Press Enter with blank textbox to exit): ")
    if user_input == '':
        print("Exited")
        break
    sentiment = predict_sentiment(user_input)
    print(f"Predicted sentiment: {sentiment}")

Model weights loaded successfully.
5093/5093 ━━━━━━━━━━━━━━━━━━━━ 4s 851us/step
FAISS index built with 162972 vectors.


Enter text (Press Enter with blank textbox to exit):  I love sunny weather


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
DEBUG: Predicted class = 2, Probabilities = [[3.0538541e-07 4.9218502e-06 9.9999475e-01]]
Similar texts retrieved from FAISS vector database:
Text: 'love modi jee', Sentiment: Positive
Text: 'superb encounter', Sentiment: Positive
Text: 'replied with love letter', Sentiment: Positive
Predicted sentiment: Positive


Enter text (Press Enter with blank textbox to exit):  I hate the service provided


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
DEBUG: Predicted class = 0, Probabilities = [[9.9999833e-01 4.8303701e-07 1.1876983e-06]]
Similar texts retrieved from FAISS vector database:
Text: 'why christians hate modi', Sentiment: Negative
Text: 'why modi hate christians', Sentiment: Negative
Text: 'morche failed modi', Sentiment: Negative
Predicted sentiment: Negative


Enter text (Press Enter with blank textbox to exit):  ok


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
DEBUG: Predicted class = 1, Probabilities = [[1.0105923e-06 9.9999583e-01 3.1856839e-06]]
Similar texts retrieved from FAISS vector database:
Text: '', Sentiment: Neutral
Text: '', Sentiment: Neutral
Text: '     ', Sentiment: Neutral
Predicted sentiment: Neutral


Enter text (Press Enter with blank textbox to exit):  


Exited
